In [1]:
import pandas as pd
from transformers import pipeline

In [2]:
# 1. CREATE FREE-TEXT SUPPORT TICKET DATASET
# ==========================================
print("Generating Free-text Support Ticket Dataset...")

Generating Free-text Support Ticket Dataset...


In [3]:
# A mock dataset representing typical IT/software help desk issues
tickets_data = {
    "ticket_id": [101, 102, 103, 104],
    "text": [
        "My screen goes completely black every time I try to open the payment page on your website. I am using Chrome on Windows.",
        "I need to update my billing address and change my credit card on file because my old one expired yesterday.",
        "Can you help me reset my password? I am locked out of my corporate portal account and didn't receive the SMS verification code.",
        "The mobile app keeps crashing immediately after the loading screen since the latest version update this morning."
    ]
}

In [4]:
df_tickets = pd.DataFrame(tickets_data)

# Define the master categories (tags) the system can choose from
candidate_tags = ["Bug Report", "Billing & Payment", "Account Security", "UI/UX Layout", "Feature Request"]

In [5]:
# 2. SETUP ZERO-SHOT LLM CLASSIFIER
# ==========================================
print("Loading zero-shot classification LLM pipeline...")
# Using Facebook's BART large MNLI model for zero-shot text classification
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

Loading zero-shot classification LLM pipeline...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [6]:
# 3. RUN INFERENCE & EXTRACT TOP 3 TAGS
# ==========================================
print("Analyzing tickets and extracting top 3 probable tags...")

Analyzing tickets and extracting top 3 probable tags...


In [7]:
all_top_3_tags = []

for index, row in df_tickets.iterrows():
    # Execute the zero-shot instruction prompt implicitly via the pipeline interface
    result = classifier(row['text'], candidate_labels=candidate_tags)

    # Extract the top 3 highest-scoring tags according to the LLM probabilities
    top_3 = result['labels'][:3]
    all_top_3_tags.append(", ".join(top_3))

In [8]:
# Append the ranked predictions back to our dataframe layout
df_tickets['Top 3 Probable Tags'] = all_top_3_tags

In [9]:
# 4. PRINT VISUAL RESULTS TABLE
# ==========================================
print("\n--- LLM Auto-Tagging Evaluation Summary ---")
pd.set_option('display.max_colwidth', None)
print(df_tickets[['ticket_id', 'text', 'Top 3 Probable Tags']])


--- LLM Auto-Tagging Evaluation Summary ---
   ticket_id  \
0        101   
1        102   
2        103   
3        104   

                                                                                                                              text  \
0         My screen goes completely black every time I try to open the payment page on your website. I am using Chrome on Windows.   
1                      I need to update my billing address and change my credit card on file because my old one expired yesterday.   
2  Can you help me reset my password? I am locked out of my corporate portal account and didn't receive the SMS verification code.   
3                 The mobile app keeps crashing immediately after the loading screen since the latest version update this morning.   

                                    Top 3 Probable Tags  
0        Bug Report, Billing & Payment, Feature Request  
1  Billing & Payment, Account Security, Feature Request  
2         Account Security, B

In [10]:
# Save output to a CSV file for your GitHub repository tracking
df_tickets.to_csv("auto_tagged_support_tickets.csv", index=False)
print("\nSuccess! Auto-tagged results exported to 'auto_tagged_support_tickets.csv'.")


Success! Auto-tagged results exported to 'auto_tagged_support_tickets.csv'.
